In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import ndcg_score
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMRanker, early_stopping, log_evaluation
import joblib
import os # For creating directories
from configuration import Config

# Utility functions

In [2]:
def load_dataset(filename: str) -> pd.DataFrame:
    """Loads a dataset from a CSV file."""
    print(f"Loading dataset: {filename}")
    return pd.read_csv(filename)

In [3]:
def print_dataframe_stats(df: pd.DataFrame, df_name: str = "DataFrame"):
    """Prints basic statistics about a DataFrame."""
    print(f"\n--- Stats for {df_name} ---")
    print(f"Shape: {df.shape}")
    print(f"Number of rows with missing values: {df.isnull().any(axis=1).sum()}")
    print(f"Number of columns with missing values: {df.isnull().any(axis=0).sum()}")
    print("---------------------------\n")

In [4]:
def calculate_mean_ndcg(
    df_with_preds: pd.DataFrame,
    true_label_col: str,
    pred_label_col: str,
    group_col: str,
    k: int
) -> float:
    """Computes the mean NDCG@k score for a ranked list."""
    ndcg_scores = []
    for _, group_df in df_with_preds.groupby(group_col):
        if len(group_df) < 1: # Changed from k to 1, as ndcg_score handles k
            continue
        true_relevance = group_df[true_label_col].values.reshape(1, -1)
        predicted_scores = group_df[pred_label_col].values.reshape(1, -1)
        
        # Ensure k is not larger than the number of items in the group if group size < k
        current_k = min(k, len(group_df))
        if current_k == 0: continue
        score = ndcg_score(true_relevance, predicted_scores, k=current_k)
        ndcg_scores.append(score)
    return np.mean(ndcg_scores) if ndcg_scores else 0.0

In [5]:
def get_group_counts(group_series: pd.Series) -> np.ndarray:
    """Calculates the size of each group."""
    return group_series.value_counts().sort_index().to_numpy()

# Models Utilities

## XGBoost Utilities

In [6]:
def create_xgb_dmatrix(X: pd.DataFrame, y: pd.Series = None, group_counts: np.ndarray = None, enable_categorical: bool = False) -> xgb.DMatrix:
    """Creates an XGBoost DMatrix."""
    # XGBoost works best if categorical features are explicitly typed as 'category' in pandas DataFrame
    # Or, use enable_categorical=True (requires XGBoost >= 1.3.0)
    # For simplicity, we assume features are numeric or XGBoost handles them with enable_categorical.
    # If enable_categorical=True, ensure your categorical columns are of dtype 'category' in X.
    # Example: for col in cat_features: X[col] = X[col].astype('category')
    dmatrix = xgb.DMatrix(X, label=y, enable_categorical=enable_categorical)
    if group_counts is not None:
        dmatrix.set_group(group_counts)
    return dmatrix

In [7]:
def train_xgb_ranker(
    params: dict,
    dtrain: xgb.DMatrix,
    num_boost_round: int,
    evals: list = None, # List of tuples (DMatrix, name) e.g., [(dtrain, 'train'), (dval, 'val')]
    early_stopping_rounds: int = None,
    verbose_eval: int = 50
) -> xgb.Booster:
    """Trains an XGBoost ranking model."""
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=evals if evals else [],
        early_stopping_rounds=early_stopping_rounds if early_stopping_rounds and evals else None,
        verbose_eval=verbose_eval
    )
    return model

## Light GBM Utilities

In [8]:
def train_lgbm_ranker(
    params: dict,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    train_group_counts: np.ndarray,
    X_val: pd.DataFrame = None,
    y_val: pd.Series = None,
    val_group_counts: np.ndarray = None,
    early_stopping_rounds: int = None,
    verbose_eval: int = 50
) -> lgb.LGBMRanker:
    """Trains a LightGBM ranking model."""
    model = LGBMRanker(**params)
    
    eval_sets = []
    eval_groups = []
    callbacks = []
    if X_val is not None and y_val is not None and val_group_counts is not None:
        eval_sets.append((X_val, y_val))
        eval_groups.append(val_group_counts)
        if early_stopping_rounds:
            callbacks.append(early_stopping(stopping_rounds=early_stopping_rounds, verbose=verbose_eval > 0))
    
    if verbose_eval > 0:
        callbacks.append(log_evaluation(period=verbose_eval))
    model.fit(
        X_train, y_train,
        group=train_group_counts,
        eval_set=eval_sets if eval_sets else None,
        eval_group=eval_groups if eval_groups else None,
        callbacks=callbacks if callbacks else None
    )
    return model

# Model training and evaluation functions

In [9]:
def get_features_target_groups(df: pd.DataFrame, config: Config) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Extracts features (X), target (y), and groups from a DataFrame."""
    y = df[config.TARGET_COL]
    # Keep PROP_ID_COL in X if it's needed for merging results later, but it's not a feature.
    # For actual training, PROP_ID_COL should be dropped if it's not a feature.
    # Let's assume X should not contain GROUP_COL, TARGET_COL, or PROP_ID_COL for training.
    feature_cols = [col for col in df.columns if col not in [config.TARGET_COL, config.GROUP_COL, config.PROP_ID_COL]]
    X = df[feature_cols]
    groups = df[config.GROUP_COL]
    return X, y, groups

In [12]:
def train_final_xgb_model(full_train_df: pd.DataFrame, config: Config) -> xgb.Booster:
    print("\n--- Training Final XGBoost Model on Full Processed Training Data ---")
    X_train, y_train, groups_train_srch = get_features_target_groups(full_train_df, config)
    train_group_counts = get_group_counts(groups_train_srch)
    dtrain = create_xgb_dmatrix(X_train, y_train, train_group_counts)
    
    # For the final model, we might not use early stopping unless we have a holdout set
    # or we use the parameters found during tuning (including num_boost_round from early stopping).
    # Here, let's assume num_boost_round is set appropriately.
    final_xgb_params = config.XGB_PARAMS.copy()
    # If best_iteration was found during tuning with early stopping, use it here.
    # e.g., final_xgb_params['num_boost_round'] = best_iteration_from_tuning
    
    xgb_model_final = train_xgb_ranker(
        params=final_xgb_params,
        dtrain=dtrain,
        num_boost_round=config.XGB_NUM_BOOST_ROUND, # Or best iteration
        evals=[(dtrain, 'train')], # Monitor training performance
        early_stopping_rounds=None, # Typically no early stopping on full data unless using a fixed num_boost_round
        verbose_eval=100
    )
    xgb_model_final.save_model(config.XGB_MODEL_PATH)
    print(f"Final XGBoost model saved to {config.XGB_MODEL_PATH}")
    return xgb_model_final

In [15]:
def train_final_lgbm_model(full_train_df: pd.DataFrame, config: Config) -> lgb.LGBMRanker:
    print("\n--- Training Final LightGBM Model on Full Processed Training Data ---")
    X_train, y_train, groups_train_srch = get_features_target_groups(full_train_df, config)
    train_group_counts = get_group_counts(groups_train_srch)
    
    final_lgbm_params = config.LGBM_PARAMS.copy()
    # If best_iteration (n_estimators) was found, set it.
    # e.g., final_lgbm_params['n_estimators'] = best_n_estimators_from_tuning
    lgbm_model_final = train_lgbm_ranker(
        params=final_lgbm_params,
        X_train=X_train, y_train=y_train, train_group_counts=train_group_counts,
        early_stopping_rounds=None, # Typically no early stopping on full data
        verbose_eval=100
    )
    lgbm_model_final.booster_.save_model(config.LGBM_MODEL_PATH)
    print(f"Final LightGBM model saved to {config.LGBM_MODEL_PATH}")
    return lgbm_model_final

# Meta - Model (Stacking)

In [13]:
def generate_oof_predictions(
    X_full: pd.DataFrame, y_full: pd.Series, groups_full_srch: pd.Series,
    model_type: str, # 'xgb' or 'lgbm'
    config: Config
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generates Out-of-Fold (OOF) predictions using GroupKFold.
    Returns: oof_preds, oof_true_labels (aligned with original X_full order)
    """
    oof_preds = np.zeros(len(X_full))
    oof_indices = np.array([]) # To store indices in their original order
    kf = GroupKFold(n_splits=config.KFOLD_N_SPLITS)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_full, y_full, groups=groups_full_srch)):
        print(f"Meta-Model OOF: Fold {fold + 1}/{config.KFOLD_N_SPLITS}")
        
        X_train_fold, X_val_fold = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_train_fold, y_val_fold = y_full.iloc[train_idx], y_full.iloc[val_idx]
        groups_train_fold_srch = groups_full_srch.iloc[train_idx]
        # groups_val_fold_srch = groups_full_srch.iloc[val_idx] # Not needed for prediction with DMatrix
        train_fold_group_counts = get_group_counts(groups_train_fold_srch)
        # val_fold_group_counts = get_group_counts(groups_val_fold_srch) # For LGBM eval_group if used
        if model_type == 'xgb':
            dtrain_fold = create_xgb_dmatrix(X_train_fold, y_train_fold, train_fold_group_counts)
            # No validation set for early stopping here to keep it simple, or use a sub-split of train_fold
            model_fold = train_xgb_ranker(config.XGB_PARAMS, dtrain_fold, config.XGB_NUM_BOOST_ROUND, verbose_eval=0) # verbose_eval=0 for less output
            dval_fold_pred = create_xgb_dmatrix(X_val_fold)
            fold_preds = model_fold.predict(dval_fold_pred)
        elif model_type == 'lgbm':
            # LGBM early stopping needs a validation set. For OOF, typically train till fixed rounds or on full fold data.
            # Or, could split train_idx further for an early stopping val set within the fold.
            # For simplicity, train for fixed n_estimators.
            lgbm_params_fold = config.LGBM_PARAMS.copy()
            # lgbm_params_fold['n_estimators'] = 200 # Example: fixed estimators for OOF
            model_fold = train_lgbm_ranker(lgbm_params_fold, X_train_fold, y_train_fold, train_fold_group_counts, verbose_eval=0)
            fold_preds = model_fold.predict(X_val_fold)
        else:
            raise ValueError("Unsupported model_type for OOF generation.")
            
        oof_preds[val_idx] = fold_preds
        oof_indices = np.concatenate([oof_indices, val_idx])

    # Ensure oof_preds are in the original order of X_full
    # This is implicitly handled if val_idx are used directly to assign oof_preds.
    # We also need y_full in the original order, which it already is.
    return oof_preds, y_full.values # y_full.values will align with X_full

In [14]:
def train_and_save_meta_model(full_train_df: pd.DataFrame, config: Config):
    print("\n--- Training Meta-Model ---")
    X_full, y_full, groups_full_srch = get_features_target_groups(full_train_df, config)
    print("Generating OOF predictions for XGBoost...")
    oof_preds_xgb, _ = generate_oof_predictions(X_full, y_full, groups_full_srch, 'xgb', config)
    
    print("Generating OOF predictions for LightGBM...")
    oof_preds_lgbm, oof_true_y = generate_oof_predictions(X_full, y_full, groups_full_srch, 'lgbm', config)
    meta_X = np.vstack([oof_preds_xgb, oof_preds_lgbm]).T
    
    # Scale meta-features
    scaler = MinMaxScaler()
    meta_X_scaled = scaler.fit_transform(meta_X)
    joblib.dump(scaler, config.META_SCALER_PATH)
    print(f"Meta-model scaler saved to {config.META_SCALER_PATH}")

    # Train meta-model (e.g., Ridge regression or a simple ranker)
    meta_model = Ridge(random_state=config.RANDOM_STATE_KFOLD) # Or another model
    meta_model.fit(meta_X_scaled, oof_true_y)
    joblib.dump(meta_model, config.META_MODEL_PATH)
    print(f"Meta-model saved to {config.META_MODEL_PATH}")

    # Evaluate meta-model on OOF predictions (optional, gives an idea of performance)
    oof_meta_predictions = meta_model.predict(meta_X_scaled)
    eval_meta_df = pd.DataFrame({
        config.GROUP_COL: groups_full_srch.values, # Ensure alignment
        config.TARGET_COL: oof_true_y,
        'predictions': oof_meta_predictions
    })
    ndcg_meta_oof = calculate_mean_ndcg(eval_meta_df, config.TARGET_COL, 'predictions', config.GROUP_COL, config.NDCG_K)
    print(f"Meta-Model OOF NDCG@{config.NDCG_K}: {ndcg_meta_oof:.5f}")
    
    return meta_model, scaler

# Submission Generator

In [16]:
def generate_submission_file(
    test_df_with_ids: pd.DataFrame, # Should contain GROUP_COL and PROP_ID_COL
    predictions: np.ndarray,
    output_filename: str,
    config: Config
):
    """Generates a submission file in the required format."""
    submission_df = test_df_with_ids[[config.GROUP_COL, config.PROP_ID_COL]].copy()
    submission_df['predicted_relevance'] = predictions
    
    # Sort by group_col (srch_id) and then by predicted_relevance (descending)
    submission_df_sorted = submission_df.sort_values(
        by=[config.GROUP_COL, 'predicted_relevance'],
        ascending=[True, False]
    )
    
    # Keep only srch_id and prop_id for the final submission file
    final_submission_df = submission_df_sorted[[config.GROUP_COL, config.PROP_ID_COL]]
    
    full_output_path = os.path.join(config.SUBMISSIONS_DIR, output_filename)
    final_submission_df.to_csv(full_output_path, index=False)
    print(f"Submission file generated: {full_output_path}")

In [17]:
def run_submission_pipeline(processed_test_df: pd.DataFrame, config: Config):
    """
    Loads trained models and generates predictions and submission files for various strategies.
    processed_test_df should be the output from preprocess_data, containing all necessary features.
    """
    print("\n--- Running Submission Pipeline ---")
    
    # Load final trained models
    try:
        xgb_model_final = xgb.Booster()
        xgb_model_final.load_model(config.XGB_MODEL_PATH)
        print("Loaded final XGBoost model.")
    except Exception as e:
        print(f"Error loading XGBoost model: {e}. Please train it first.")
        return
    try:
        lgbm_model_final = lgb.Booster(model_file=config.LGBM_MODEL_PATH)
        print("Loaded final LightGBM model.")
    except Exception as e:
        print(f"Error loading LightGBM model: {e}. Please train it first.")
        return
        
    try:
        meta_model = joblib.load(config.META_MODEL_PATH)
        meta_scaler = joblib.load(config.META_SCALER_PATH)
        print("Loaded meta-model and scaler.")
    except Exception as e:
        print(f"Error loading meta-model/scaler: {e}. Please train them first.")
        return

    # Prepare test data features (X_test)
    # Ensure PROP_ID_COL is not in X_test for prediction if it's not a feature
    feature_cols_test = [col for col in processed_test_df.columns if col not in [config.GROUP_COL, config.TARGET_COL, config.PROP_ID_COL]]
    X_test_submission = processed_test_df[feature_cols_test]

    # 1. XGBoost Predictions
    print("Generating XGBoost predictions for submission...")
    dtest_submission = create_xgb_dmatrix(X_test_submission)
    preds_xgb_submission = xgb_model_final.predict(dtest_submission)
    generate_submission_file(processed_test_df, preds_xgb_submission, "submission_xgboost.csv", config)

    # 2. LightGBM Predictions
    print("Generating LightGBM predictions for submission...")
    preds_lgbm_submission = lgbm_model_final.predict(X_test_submission)
    generate_submission_file(processed_test_df, preds_lgbm_submission, "submission_lightgbm.csv", config)

    # 3. Ensemble Predictions (Weighted Average)
    print("Generating Ensemble (XGB+LGBM) predictions for submission...")
    
    # Simple weighted average without scaling:
    ensemble_preds = (config.ENSEMBLE_XGB_WEIGHT * preds_xgb_submission +
                      config.ENSEMBLE_LGB_WEIGHT * preds_lgbm_submission)
    generate_submission_file(processed_test_df, ensemble_preds, "submission_ensemble_weighted.csv", config)
    
    # 4. Meta-Model Predictions
    print("Generating Meta-Model predictions for submission...")
    meta_X_submission = np.vstack([preds_xgb_submission, preds_lgbm_submission]).T
    meta_X_submission_scaled = meta_scaler.transform(meta_X_submission) # Use the scaler fitted during meta-training
    meta_preds_submission = meta_model.predict(meta_X_submission_scaled)
    generate_submission_file(processed_test_df, meta_preds_submission, "submission_meta_model.csv", config)
    print("Submission pipeline complete.")

# Main execution

In [ ]:
if __name__ == "__main__":
    config = Config()
    Config.create_dirs() # Ensure directories exist

    train_models = True

    # --- MAKE SURE TO RUN THE FEATURE ENGINEERING NOTEBOOK FIRST ---
    try:
        train_processed_df = load_dataset(config.PROCESSED_TRAINING_FILE)
        test_processed_df = load_dataset(config.PROCESSED_TEST_FILE) # This is for final submission
        print("Loaded preprocessed training and test data.")
        print_dataframe_stats(train_processed_df, "Loaded Processed Training Data")
        print_dataframe_stats(test_processed_df, "Loaded Processed Test Data for Submission")
    except FileNotFoundError:
        print("Processed files not found. Please run the preprocessing notebook first.")

    if train_models == True:
        train_final_xgb_model(train_processed_df, config)
        train_final_lgbm_model(train_processed_df, config)
        # Train meta-model (uses KFold on full processed training data)
        train_and_save_meta_model(train_processed_df, config)
        print("Final models training complete.")

        
    run_submission_pipeline(test_processed_df, config)



Loading dataset: ./datasets\processed_data\train_processed.csv


Loading dataset: ./datasets\processed_data\test_processed.csv
Loaded preprocessed training and test data.

--- Stats for Loaded Processed Training Data ---
Shape: (3722553, 141)
Number of rows with missing values: 0
Number of columns with missing values: 0
---------------------------


--- Stats for Loaded Processed Test Data for Submission ---
Shape: (4959183, 140)
Number of rows with missing values: 0
Number of columns with missing values: 0
---------------------------


--- Training Final XGBoost Model on Full Processed Training Data ---
[0]	train-ndcg@5:0.28318
[100]	train-ndcg@5:0.39387
[200]	train-ndcg@5:0.40455
[300]	train-ndcg@5:0.41352
[400]	train-ndcg@5:0.42091
[500]	train-ndcg@5:0.42784
[600]	train-ndcg@5:0.43419
[700]	train-ndcg@5:0.43955
[800]	train-ndcg@5:0.44470
[900]	train-ndcg@5:0.44990
[999]	train-ndcg@5:0.45411
Final XGBoost model saved to ./datasets\models\xgb_ranker_final.json

--- Training Final LightGBM Model on Full Processed Training Data ---


c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Final LightGBM model saved to ./datasets\models\lgbm_ranker_final.txt

--- Training Meta-Model ---
Generating OOF predictions for XGBoost...
Meta-Model OOF: Fold 1/5
Meta-Model OOF: Fold 2/5
Meta-Model OOF: Fold 3/5
Meta-Model OOF: Fold 4/5
Meta-Model OOF: Fold 5/5
Generating OOF predictions for LightGBM...
Meta-Model OOF: Fold 1/5


c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


...

Meta-Model OOF: Fold 4/5


c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Meta-Model OOF: Fold 5/5


c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
c:\Users\andrea\miniconda3\envs\datamining\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Meta-model scaler saved to ./datasets\models\meta_scaler.pkl
Meta-model saved to ./datasets\models\meta_model_ridge.pkl
Meta-Model OOF NDCG@5: 0.40322
Final models training complete.

--- Running Submission Pipeline ---
Loaded final XGBoost model.
Loaded final LightGBM model.
Loaded meta-model and scaler.
Generating XGBoost predictions for submission...
Submission file generated: ./datasets\submissions\submission_xgboost.csv
Generating LightGBM predictions for submission...
Submission file generated: ./datasets\submissions\submission_lightgbm.csv
Generating Ensemble (XGB+LGBM) predictions for submission...
Submission file generated: ./datasets\submissions\submission_ensemble_weighted.csv
Generating Meta-Model predictions for submission...
Submission file generated: ./datasets\submissions\submission_meta_model.csv
Submission pipeline complete.
